# Image Transport

> How to use ROS2 `image_transport` to send compressed images and fix bandwidth problems — especially in WSL.

## The Problem

Sending raw uncompressed images over ROS2 topics is expensive. A single 640×480 BGR frame is ~900 KB. At 30 FPS that's ~27 MB/s — enough to saturate a USB 2.0 connection or cause timeouts in WSL where the virtual bridge has limited bandwidth.

The `image_transport` package solves this by providing **transparent compression plugins**. You change one line of code and the system automatically compresses/decompresses images on the wire.

## What is `image_transport`?

`image_transport` is a ROS2 package that provides transparent support for transporting images in low-bandwidth compressed formats. It wraps publishers and subscribers so they can automatically use different compression methods.

Key features:

- **Drop-in replacement** for `Publisher` and `Subscription` in image nodes
- **Plugin architecture** — compression plugins are loaded at runtime
- **Topic remapping** — each transport creates its own sub-topic (e.g. `/camera/image_raw/compressed`)
- **Zero code changes for subscribers** — they auto-detect available transports

## Available Plugins

Install the plugins with:

```sh
sudo apt install ros-$ROS_DISTRO-image-transport-plugins
```

| Plugin | Compression | Quality | Bandwidth | Use case |
|--------|-------------|---------|-----------|----------|
| `raw` | None | Lossless | ~27 MB/s (640×480@30fps) | Local development, high-bandwidth networks |
| `compressed` | JPEG/PNG | Lossy/lossless | ~1-3 MB/s | General purpose, most common |
| `theora` | Theora video | Lossy | ~0.5-2 MB/s | Streaming video, long-running feeds |
| `compressed_depth` | PNG (16-bit) | Lossless | Variable | Depth cameras (RealSense, Kinect) |

## Using `image_transport` in C++

In C++ nodes, replace `rclcpp::Publisher<sensor_msgs::msg::Image>` with `image_transport::Publisher`:

```cpp
#include <image_transport/image_transport.hpp>

// Instead of:
// auto pub = create_publisher<Image>("/camera/image_raw", 10);

// Use:
auto pub = image_transport::create_publisher(this, "/camera/image_raw");

// Publishing is the same:
pub.publish(msg);
```

For subscribers:

```cpp
auto sub = image_transport::create_subscription(
    this, "/camera/image_raw",
    std::bind(&MyNode::image_callback, this, std::placeholders::_1),
    "raw"  // default transport
);
```

> **Note:** `image_transport` is primarily a C++ library. Python support is limited — in Python nodes you typically subscribe to the specific transport topic directly (e.g. `/camera/image_raw/compressed`).

## Using `image_transport` in Python

Python nodes don't have a native `image_transport` wrapper, but you can still benefit from it:

1. **Publish raw** — the C++ `image_transport` plugin on the subscriber side will auto-detect and decompress
2. **Subscribe to compressed** — subscribe directly to the compressed topic:

```python
from sensor_msgs.msg import CompressedImage

# Subscribe to compressed topic
self.sub = self.create_subscription(
    CompressedImage,
    '/camera/image_raw/compressed',
    self.image_callback,
    10
)

def image_callback(self, msg):
    import cv2
    import numpy as np
    np_arr = np.frombuffer(msg.data, np.uint8)
    frame = cv2.imdecode(np_arr, cv2.IMREAD_COLOR)
```

3. **Use `cv_bridge` with compressed** — `CvBridge` can handle `CompressedImage` too:

```python
from cv_bridge import CvBridge
bridge = CvBridge()
frame = bridge.compressed_imgmsg_to_cv2(msg)
```

## Choosing a Transport

You can set the default transport for a subscriber via ROS2 parameters or topic remapping:

```sh
# Subscribe to compressed instead of raw
ros2 run my_package my_node --ros-args -p image_transport:=compressed
```

Or remap the topic at launch:

```python
# In a launch file
Node(
    package='my_package',
    executable='my_node',
    remappings=[
        ('/camera/image_raw', '/camera/image_raw/compressed')
    ]
)
```

## WSL Bridge Problem

When running ROS2 in WSL (Windows Subsystem for Linux), the virtual network bridge between Windows and Linux has limited bandwidth. Raw image topics often cause:

- **Timeouts** — subscribers miss frames
- **Lag** — real-time visualization stutters
- **High CPU** — encoding/decoding raw frames is expensive

### Solutions

1. **Use `compressed` transport** — reduces bandwidth by 10-30x
2. **Lower resolution** — 320×240 instead of 640×480
3. **Lower FPS** — 15 FPS instead of 30
4. **MJPG format** — set the camera to output MJPG natively (already covered in notebook 02)

```sh
# Example: compressed transport at 15 FPS, 320×240
ros2 run ros2_cam_nbdev cam_pub --ros-args \
    -p frame_rate:=15.0 \
    -p resolution_width:=320 \
    -p resolution_height:=240
```

Then subscribe to `/camera/image_raw/compressed` instead of `/camera/image_raw`.

## Inspecting Transports

Check which transports are available on a topic:

```sh
ros2 topic info /camera/image_raw
Type: sensor_msgs/msg/Image

ros2 topic list | grep image_raw
/camera/image_raw
/camera/image_raw/compressed
/camera/image_raw/theora
```

View compressed images:

```sh
ros2 run rqt_image_view rqt_image_view /camera/image_raw/compressed
```

## Summary

| Action | Command/Code |
|--------|-------------|
| Install plugins | `sudo apt install ros-$ROS_DISTRO-image-transport-plugins` |
| C++ publisher | `image_transport::create_publisher(node, topic)` |
| C++ subscriber | `image_transport::create_subscription(node, topic, callback, transport)` |
| Python compressed sub | Subscribe to `/topic/compressed` with `CompressedImage` |
| Set transport param | `-p image_transport:=compressed` |
| View compressed | `rqt_image_view /topic/compressed` |

## Next Steps

Now that we can send images efficiently, let's look at camera calibration and `CameraInfo` in `06_calibration_and_info`.